# 공차 전체 음료 칼로리 분석

- **수집 대상:** https://www.gong-cha.co.kr/brand/menu/product?category=001001
- **robots.txt 확인:** `/brand/menu/` 경로는 수집 허용 (`disallow: /thdadmin/, /upload/, /html2/, /html/`)
- **목적:** 공차 음료 중 칼로리가 가장 높은/낮은 음료 Top 5 찾기 (**L 사이즈 아이스(Cold) 기준**)

### 사이트 구조 (직접 확인한 내용)
- 음료 탭(New 시즌 메뉴, 밀크티, 커피 …)은 클릭하면 주소의 `category` 번호만 바뀌는 **정적 페이지**예요.
  → Selenium 없이 `requests` + `BeautifulSoup`으로 충분해요.
- 상세 페이지 영양정보 표의 열 순서: `구분(Cold/Hot) | 사이즈(L/J) | 컵 용량(ml) | 열량(kcal) | 당류 | 단백질 | 포화지방 | 나트륨 | 카페인`

**셀을 위에서부터 순서대로 실행하세요** (`Shift + Enter`).

## 0. 라이브러리 설치 (처음 한 번만)

In [ ]:
%pip install requests beautifulsoup4 pandas

## 1. 라이브러리 불러오기 + 설정

In [3]:
import re
import time

import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE = "https://www.gong-cha.co.kr"
LIST_URL = BASE + "/brand/menu/product?category={code}"
HEADERS = {"User-Agent": "Mozilla/5.0"}

# 음료 탭 이름과 category 번호 (사이트의 탭 HTML에서 확인한 값)
# 먼저 나온 탭 이름이 그 음료의 카테고리가 되도록 진짜 종류를 앞에 뒀어요.
TABS = {
    "001006": "밀크티",
    "001010": "스무디",
    "001003": "오리지널 티",
    "001015": "프룻티&모어",
    "001011": "커피",
    "001017": "요거티",
    "001001": "New 시즌 메뉴",
    "001002": "베스트셀러",
}

## 2. 함수: 탭(카테고리) 페이지에서 메뉴 이름과 상세 링크 모으기

In [4]:
def get_menus_in_tab(code):
    res = requests.get(LIST_URL.format(code=code), headers=HEADERS, timeout=10)
    res.raise_for_status()
    soup = BeautifulSoup(res.text, "html.parser")

    items = []
    # 메뉴 하나 = <a href="/brand/menu/product_detail?category=...&no=..."> 안의 <p>메뉴이름</p>
    for a in soup.select('a[href*="product_detail"]'):
        name = " ".join(a.get_text(" ", strip=True).split())
        url = BASE + a["href"]               # 링크 주소를 그대로 사용 (category 번호 포함)
        if name:
            items.append({"name": name, "url": url})
    return items

## 3. 함수: 상세 페이지 영양정보 표에서 L 사이즈 아이스(Cold) 값 가져오기

In [5]:
def get_nutrition(url):
    res = requests.get(url, headers=HEADERS, timeout=10)
    res.raise_for_status()
    soup = BeautifulSoup(res.text, "html.parser")

    rows = []
    temp = None                                  # Cold / Hot
    for tr in soup.select("table tbody tr"):
        cells = [td.get_text(strip=True) for td in tr.find_all("td")]
        if len(cells) < 8:
            continue
        # 표에서 'Cold'/'Hot' 칸은 여러 줄을 합쳐 놓아서(rowspan) 첫 줄에만 있어요.
        # 그래서 칸이 9개면 첫 칸이 온도, 8개면 위 줄의 온도를 그대로 써요.
        if len(cells) == 9:
            temp = cells[0]
        # 뒤에서부터 세면 칸 위치가 항상 같아요:
        # [-8]사이즈 [-7]용량 [-6]열량 [-5]당류 [-4]단백질 [-3]포화지방 [-2]나트륨 [-1]카페인
        rows.append({
            "온도": temp,
            "사이즈": cells[-8],
            "용량(ml)": float(cells[-7]),
            "칼로리(kcal)": float(cells[-6]),
            "당류(g)": float(cells[-5]),
            "카페인(mg)": float(cells[-1]),
        })

    # L 사이즈 아이스(Cold) 줄 찾기 → 없으면 None
    for r in rows:
        if r["온도"] == "Cold" and r["사이즈"] == "L":
            return r
    return None

## 4. 1단계 실행: 모든 탭에서 음료 모으기 (중복 제거)

In [6]:
menus = {}                                   # {메뉴이름: 정보}  → 이름이 같으면 한 번만
for code, tab in TABS.items():
    items = get_menus_in_tab(code)
    new = 0
    for it in items:
        if it["name"] not in menus:
            menus[it["name"]] = {"name": it["name"], "category": tab, "url": it["url"]}
            new += 1
    print(f"[{tab}] {len(items)}개 (새로 추가 {new}개)")
    time.sleep(1)                            # 서버 부담 줄이기

menus = list(menus.values())
print(f"\n중복 빼고 음료 {len(menus)}개")

[밀크티] 12개 (새로 추가 12개)
[스무디] 17개 (새로 추가 17개)
[오리지널 티] 4개 (새로 추가 4개)
[프룻티&모어] 9개 (새로 추가 9개)
[커피] 6개 (새로 추가 6개)
[요거티] 4개 (새로 추가 4개)
[New 시즌 메뉴] 36개 (새로 추가 36개)
[베스트셀러] 14개 (새로 추가 13개)

중복 빼고 음료 101개


## 5. 2단계 실행: 각 음료의 영양정보 모으기
음료 하나당 1초씩 쉬어서 2분 정도 걸려요.

In [7]:
for i, m in enumerate(menus, 1):
    try:
        info = get_nutrition(m["url"])
    except Exception as e:
        print("  에러:", m["name"], e)
        info = None
    if info:
        m.update(info)
        print(f"[{i}/{len(menus)}] {m['name']}: {info['칼로리(kcal)']:.0f}kcal")
    else:
        print(f"[{i}/{len(menus)}] {m['name']}: L 사이즈 아이스 정보 없음 (제외)")
    time.sleep(1)

[1/101] 브라운슈가 시그니처 밀크티 +미니펄: 407kcal
[2/101] 미니펄 망고 밀크: 301kcal
[3/101] 미니펄 딸기 밀크티: 296kcal
[4/101] 피스타치오 밀크티: 227kcal
[5/101] 제주 그린 밀크티: 184kcal
[6/101] 브라운슈가 쥬얼리 밀크티: 328kcal
[7/101] 얼그레이 밀크티: 205kcal
[8/101] 자스민 밀크티: 206kcal
[9/101] 우롱 밀크티: 205kcal
[10/101] 블랙 밀크티: 206kcal
[11/101] 타로 밀크티: 263kcal
[12/101] 초콜렛 밀크티: 196kcal
[13/101] 타로 쥬얼리 시그니처 스무디: 467kcal
[14/101] 초코바른 초코 스무디: 606kcal
[15/101] 초코바른 제주 그린 스무디: 498kcal
[16/101] 초코바른 피스타치오 스무디: 476kcal
[17/101] 밀크 쿠앤크 스무디: 459kcal
[18/101] 브라운슈가 쥬얼리 치즈폼 스무디: 572kcal
[19/101] 미니펄 망고 크러쉬: 408kcal
[20/101] 미니펄 딸기 크러쉬: 466kcal
[21/101] 타로 스무디: 354kcal
[22/101] 레몬 요구르트 스무디: 343kcal
[23/101] 초코멜로 스무디: 460kcal
[24/101] 초콜렛 쿠키 & 크림 스무디: 516kcal
[25/101] 딸기 쿠키 스무디: 587kcal
[26/101] 딸기 쥬얼리 요구르트 크러쉬: 349kcal
[27/101] 제주 그린 스무디: 476kcal
[28/101] 망고 스무디: 407kcal
[29/101] 청포도 스무디: 329kcal
[30/101] 자스민티: 4kcal
[31/101] 우롱티: 3kcal
[32/101] 블랙티: 4kcal
[33/101] 얼그레이티: 3kcal
[34/101] 허니 자몽 블랙티: 126kcal
[35/101] 레몬 자스민티: 112kcal
[36/101] 레몬 요구르트: 212kcal

## 6. 데이터 확인 + CSV 저장

In [8]:
df = pd.DataFrame(menus)
df.to_csv("gongcha_all_drinks.csv", index=False, encoding="utf-8-sig")  # 엑셀에서 한글 안 깨지게
print("CSV 저장 완료: gongcha_all_drinks.csv")
df

CSV 저장 완료: gongcha_all_drinks.csv


,name,category,url,온도,사이즈,용량(ml),칼로리(kcal),당류(g),카페인(mg)
0,브라운슈가 시그니처 밀크티 +미니펄,밀크티,https://www.gong-cha.co.kr/brand/menu/product_...,Cold,L,473.0,407.0,40.0,92.0
1,미니펄 망고 밀크,밀크티,https://www.gong-cha.co.kr/brand/menu/product_...,Cold,L,473.0,301.0,33.0,0.0
2,미니펄 딸기 밀크티,밀크티,https://www.gong-cha.co.kr/brand/menu/product_...,Cold,L,473.0,296.0,40.0,49.0
3,피스타치오 밀크티,밀크티,https://www.gong-cha.co.kr/brand/menu/product_...,Cold,L,473.0,227.0,25.0,20.0
4,제주 그린 밀크티,밀크티,https://www.gong-cha.co.kr/brand/menu/product_...,Cold,L,473.0,184.0,16.0,49.0
...,...,...,...,...,...,...,...,...,...
96,우롱티 + 코코넛 + 밀크폼,베스트셀러,https://www.gong-cha.co.kr/brand/menu/product_...,Cold,L,473.0,137.0,13.0,64.0
97,딸기 쥬얼리 밀크티,베스트셀러,https://www.gong-cha.co.kr/brand/menu/product_...,Cold,L,473.0,225.0,26.0,29.0
98,망고 요구르트 + 화이트펄,베스트셀러,https://www.gong-cha.co.kr/brand/menu/product_...,Cold,L,473.0,209.0,36.0,0.0
99,타로 밀크티 + 펄,베스트셀러,https://www.gong-cha.co.kr/brand/menu/product_...,Cold,L,473.0,357.0,14.0,0.0


## 7. 분석: 칼로리 높은/낮은 음료 Top 5 (L 사이즈 아이스 기준)

In [9]:
data = df.dropna(subset=["칼로리(kcal)"])   # L 사이즈 아이스 정보가 없는 음료는 제외
top5 = data.sort_values("칼로리(kcal)", ascending=False, kind="stable").head(5)
low5 = data.sort_values("칼로리(kcal)", ascending=True, kind="stable").head(5)

print("[칼로리 높은 음료 Top 5] (L 사이즈 아이스 기준)")
for rank, (name, kcal) in enumerate(zip(top5["name"], top5["칼로리(kcal)"]), 1):
    print(f"{rank}등. {name} {kcal:.0f}kcal")

print("\n[칼로리 낮은 음료 Top 5] (L 사이즈 아이스 기준)")
for rank, (name, kcal) in enumerate(zip(low5["name"], low5["칼로리(kcal)"]), 1):
    print(f"{rank}등. {name} {kcal:.0f}kcal")

print(f"\n분석한 음료 {len(data)}개, 평균 칼로리 {data['칼로리(kcal)'].mean():.0f}kcal")

[칼로리 높은 음료 Top 5] (L 사이즈 아이스 기준)
1등. 딸기 쿠키 스무디 + 펄 679kcal
2등. 초코바른 초코 스무디 606kcal
3등. 딸기 쿠키 스무디 587kcal
4등. 브라운슈가 쥬얼리 치즈폼 스무디 572kcal
5등. 초콜릿 시그니처 밀크티+펄 540kcal

[칼로리 낮은 음료 Top 5] (L 사이즈 아이스 기준)
1등. 우롱티 3kcal
2등. 얼그레이티 3kcal
3등. 자스민티 4kcal
4등. 블랙티 4kcal
5등. 저당 우롱 오리지널 티 5kcal

분석한 음료 98개, 평균 칼로리 275kcal


In [14]:
# 8. 순위 출력 함수 (값이 같으면 같은 등수)
def print_rank(table, col, unit, n=5, title="", ascending=False):
    """table에서 col 값 순서로 n등까지 출력 (동점은 같은 등수)
    ascending=False → 큰 값부터(많은 순), ascending=True → 작은 값부터(적은 순)"""
    t = table.dropna(subset=[col]).copy()
    t["순위"] = t[col].rank(method="min", ascending=ascending).astype(int)   # 동점 = 같은 등수
    t = t[t["순위"] <= n].sort_values(["순위", "name"])
    print(title)
    for _, row in t.iterrows():
        print(f"{row['순위']}등. {row['name']} {row[col]:.0f}{unit}")

In [17]:
# 9-1. 당류가 가장 많은 음료 Top 5
print_rank(data, "당류(g)", "g", n=5, title="[당류 많은 음료 Top 5] (L 사이즈 아이스 기준)")

# 9-2. 당류가 0g인 음료 목록
no_sugar = data[data["당류(g)"] == 0]
print(f"\n[당류 0g 음료] {len(no_sugar)}개 / 전체 {len(data)}개")
for i, name in enumerate(no_sugar["name"], 1):
    print(f"{i}. {name}")

[당류 많은 음료 Top 5] (L 사이즈 아이스 기준)
1등. 청포도 스무디 69g
2등. 레몬 요구르트 스무디 66g
3등. 얼그레이 말차 밀크티 + 펄 63g
4등. 딸기 쿠키 스무디 + 펄 58g
5등. 포도 말랑 크러쉬 57g

[당류 0g 음료] 10개 / 전체 98개
1. 자스민티
2. 우롱티
3. 블랙티
4. 얼그레이티
5. 얼그레이 아메리카노
6. 저당 블랙 오리지널 티
7. 저당 얼그레이 오리지널 티
8. 저당 우롱 오리지널 티
9. 저당 자스민 오리지널 티
10. 아메리카노


In [12]:
# 10. 카페인이 가장 많은 음료 Top 5 + 카페인 없는 음료
print_rank(data, "카페인(mg)", "mg", n=5, title="[카페인 많은 음료 Top 5] (L 사이즈 아이스 기준)")

no_caffeine = data[data["카페인(mg)"] == 0]
print(f"\n카페인 0mg 음료: {len(no_caffeine)}개 / 전체 {len(data)}개")
print(", ".join(no_caffeine["name"]))

[카페인 많은 음료 Top 5] (L 사이즈 아이스 기준)
1등. 얼그레이 말차 밀크티 + 펄 186mg
2등. 공차슈페너 178mg
3등. 얼그레이 밀크티 175mg
3등. 저당 얼그레이 밀크티 175mg
3등. 저당 얼그레이 밀크티 + 펄 175mg
3등. 포도 말랑 밀크티 175mg

카페인 0mg 음료: 21개 / 전체 98개
미니펄 망고 밀크, 브라운슈가 쥬얼리 밀크티, 타로 밀크티, 타로 쥬얼리 시그니처 스무디, 밀크 쿠앤크 스무디, 브라운슈가 쥬얼리 치즈폼 스무디, 미니펄 망고 크러쉬, 미니펄 딸기 크러쉬, 타로 스무디, 레몬 요구르트 스무디, 망고 스무디, 레몬 요구르트, 자몽 요구르트, 패션 프룻 히비스커스, 망고 주스, 망고 요구르트, 코코 타로 쥬얼리 밀크티, 타로 쥬얼리 시그니처 밀크티, 미니펄 망고 밀크 + 리얼 망고, 망고 요구르트 + 화이트펄, 타로 밀크티 + 펄


In [16]:
# 11. 베스트셀러 중 칼로리 Top 3
best_names = [it["name"] for it in get_menus_in_tab("001002")]   # 베스트셀러 탭 메뉴 이름
best = data[data["name"].isin(best_names)]
print(f"베스트셀러 {len(best_names)}개 중 분석 가능한 음료 {len(best)}개\n")

# 11-1. 베스트셀러 중 칼로리 높은 Top 3
print_rank(best, "칼로리(kcal)", "kcal", n=3, title="[베스트셀러 칼로리 높은 Top 3] (L 사이즈 아이스 기준)")

# 11-2. 베스트셀러 중 칼로리 낮은 Top 3
print_rank(best, "칼로리(kcal)", "kcal", n=3, ascending=True,
           title="\n[베스트셀러 칼로리 낮은 Top 3] (L 사이즈 아이스 기준)")

베스트셀러 14개 중 분석 가능한 음료 14개

[베스트셀러 칼로리 높은 Top 3] (L 사이즈 아이스 기준)
1등. 딸기 쿠키 스무디 + 펄 679kcal
2등. 브라운슈가 시그니처 밀크티 +펄 424kcal
3등. 딸기 쥬얼리 시그니처 밀크티 386kcal

[베스트셀러 칼로리 낮은 Top 3] (L 사이즈 아이스 기준)
1등. 아메리카노 10kcal
2등. 허니 자몽 블랙티 126kcal
3등. 우롱티 + 코코넛 + 밀크폼 137kcal
